# 02 — Combine Aggregated Classes into the Unified Split Dataset

Merges the four `data/aggregated/<class>/` folders into one stratified 70/20/10 split at `data/dataset/`. Assigns a deterministic global ID (1–800) per image and renames to `<NNNN>_<class>.jpg`. Remaps each label's `class_id` from `0` to the global index (`projector=0, whiteboard=1, fire_extinguisher=2, door_sign=3`).

In [ ]:
# Install dependencies
%pip install -q pandas pillow matplotlib pyyaml

In [ ]:
# Imports
import random, shutil
from pathlib import Path
import pandas as pd
import yaml
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Config — paths, classes, split ratios
AGG = Path('../data/aggregated')   # input: per-class aggregated folders from notebook 01
DST = Path('../data/dataset')      # output: unified split dataset

# Class order defines the global class_id (0..3)
CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']
GLOBAL_ID = {c: i for i, c in enumerate(CLASSES)}

SPLIT_RATIOS = {'train': 0.70, 'val': 0.20, 'test': 0.10}
SEED = 42

random.seed(SEED)
for s in SPLIT_RATIOS:
    (DST / 'images' / s).mkdir(parents=True, exist_ok=True)
    (DST / 'labels' / s).mkdir(parents=True, exist_ok=True)

## 1. Discover aggregated pairs per class

In [ ]:
# List (image, label) pairs from data/aggregated/<class>/
pairs_per_class = {}
for c in CLASSES:
    imgs = sorted((AGG / c / 'images').glob('*.jpg'))
    pairs = [(img, AGG / c / 'labels' / (img.stem + '.txt')) for img in imgs]
    pairs_per_class[c] = [(i, l) for i, l in pairs if l.exists()]

pd.DataFrame([(c, len(p)) for c, p in pairs_per_class.items()],
             columns=['class', 'available_pairs'])

## 2. Assign global IDs (1–800), then stratified per-class split
Per-class shuffle guarantees every class lands in every split. Global IDs are deterministic, so the namespace is reproducible across runs.

In [ ]:
# Build the unified pool, assign global_id 1..N, then split per class
unified = []
for c in CLASSES:
    for img, lbl in pairs_per_class[c]:
        per_class_id = int(img.stem.rsplit('_', 1)[-1])  # 'projector_0042' -> 42
        unified.append({'class': c, 'img': img, 'lbl': lbl, 'per_class_id': per_class_id})
unified.sort(key=lambda r: (CLASSES.index(r['class']), r['per_class_id']))
for gid, r in enumerate(unified, 1):
    r['global_id'] = gid

assignments = []
for c in CLASSES:
    pool = [r for r in unified if r['class'] == c]
    random.shuffle(pool)
    n = len(pool)
    n_tr = int(n * SPLIT_RATIOS['train'])
    n_va = int(n * SPLIT_RATIOS['val'])
    for r in pool[:n_tr]:           assignments.append({**r, 'split': 'train'})
    for r in pool[n_tr:n_tr+n_va]:  assignments.append({**r, 'split': 'val'})
    for r in pool[n_tr+n_va:]:      assignments.append({**r, 'split': 'test'})

plan = pd.DataFrame(assignments)
plan.groupby(['split', 'class']).size().unstack(fill_value=0).reindex(['train','val','test'])

## 3. Copy with global naming, remap `class_id`, write `manifest.csv`
Files written as `<NNNN>_<class>.jpg` under `images/<split>/`. Split is encoded by directory, so it's not in the filename.

In [ ]:
# Copy + class_id remap; emit manifest.csv joining global_id <-> per_class_id
manifest = []
for r in assignments:
    stem = f"{r['global_id']:04d}_{r['class']}"
    dst_img = DST / 'images' / r['split'] / f'{stem}.jpg'
    dst_lbl = DST / 'labels' / r['split'] / f'{stem}.txt'
    shutil.copy2(r['img'], dst_img)

    cid = GLOBAL_ID[r['class']]
    remapped = []
    for line in r['lbl'].read_text().strip().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        _, xc, yc, w, h = parts
        remapped.append(f'{cid} {xc} {yc} {w} {h}')
    dst_lbl.write_text('\n'.join(remapped) + ('\n' if remapped else ''))

    manifest.append({
        'filename': dst_img.name,
        'global_id': r['global_id'],
        'per_class_id': r['per_class_id'],
        'class': r['class'],
        'class_id': cid,
        'split': r['split'],
        'boxes': len(remapped),
    })

mf = pd.DataFrame(manifest).sort_values('global_id').reset_index(drop=True)
mf.to_csv(DST / 'manifest.csv', index=False)
mf.groupby(['split', 'class']).agg(images=('filename','count'), boxes=('boxes','sum'))

## 4. Structural validation
Every image has a matching label; rows have 5 fields; `class_id ∈ [0,3]`; coords in `[0,1]`.

In [ ]:
# Walk every label, collect any structural issues
issues = []
for split in SPLIT_RATIOS:
    img_dir = DST / 'images' / split; lbl_dir = DST / 'labels' / split
    for img in img_dir.glob('*.jpg'):
        lbl = lbl_dir / (img.stem + '.txt')
        if not lbl.exists():
            issues.append((split, img.name, 'missing_label')); continue
        for ln, line in enumerate(lbl.read_text().strip().splitlines(), 1):
            parts = line.split()
            if len(parts) != 5:
                issues.append((split, img.name, f'bad_row_{ln}')); continue
            cid, *coords = parts
            if int(cid) not in range(len(CLASSES)):
                issues.append((split, img.name, f'bad_class_{cid}'))
            if any(not (0 <= float(x) <= 1) for x in coords):
                issues.append((split, img.name, f'coords_out_of_range_row_{ln}'))

pd.DataFrame(issues, columns=['split', 'file', 'issue']).head(20)

## 5. Visual spot-check — confirm the class remap

In [ ]:
# Overlay 3 random train images per class, with predicted-class labels
def draw(ax, img_path, lbl_path):
    im = Image.open(img_path); W, H = im.size
    ax.imshow(im); ax.set_title(img_path.name, fontsize=8); ax.axis('off')
    for line in lbl_path.read_text().strip().splitlines():
        cid, xc, yc, w, h = map(float, line.split())
        x = (xc - w/2) * W; y = (yc - h/2) * H
        ax.add_patch(patches.Rectangle((x, y), w*W, h*H, fill=False, linewidth=1.5, edgecolor='lime'))
        ax.text(x, y-3, CLASSES[int(cid)], color='lime', fontsize=7)

train_img = DST / 'images' / 'train'; train_lbl = DST / 'labels' / 'train'
fig, axes = plt.subplots(len(CLASSES), 3, figsize=(12, 4 * len(CLASSES)))
for row, c in enumerate(CLASSES):
    picks = random.sample(list(train_img.glob(f'*_{c}.jpg')), 3)
    for ax, p in zip(axes[row], picks):
        draw(ax, p, train_lbl / (p.stem + '.txt'))
plt.tight_layout(); plt.show()

## 6. Emit `data.yaml` for Ultralytics

In [ ]:
# Write data/dataset/data.yaml
cfg = {
    'path': str(DST.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'names': {i: c for i, c in enumerate(CLASSES)},
}
(DST / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print((DST / 'data.yaml').read_text())

**Done.** Stratified split is on disk with consistent naming and a joinable manifest. Continue to **notebook 03** for dataset-health diagnostics.